# Space Cadet: PLAN-compliant re-entry classification

This notebook uses the separate `data/processed_plan/panel.csv` built by `src/build_plan_dataset.py`. It follows PLAN.md: junk object types only; monthly snapshots from January 2015 through the observed as-of date minus 365 days; orbit measured as of each snapshot, perigee at most 600 km, element data no older than 30 days; observed-only trailing space weather from the prior day; and seeded object-disjoint cohorts restricted to train 2015–2020, validation 2022, test 2024 onward (2021 and 2023 omitted).

Unlike the original notebook, this dataset retains eligible objects without a recorded re-entry when they have a full 365 days of follow-up. `reenters_within_horizon` is the classification target. The test split is still a development holdout because earlier versions of this panel were evaluated; it is not an untouched final test.


## Shared data, split checks, and evaluation helpers


In [5]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, average_precision_score,
                             roc_auc_score, precision_score, recall_score,
                             brier_score_loss, confusion_matrix)
from threadpoolctl import threadpool_limits

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "data/processed_plan/panel.csv").is_file())
plan_dir = PROJECT_ROOT / "data/processed_plan"
data = pd.read_csv(plan_dir / "panel.csv", parse_dates=["obs_date"])
TARGET = "reenters_within_horizon"
SEED = 20261021
feature_groups = json.loads((plan_dir / "feature_groups.json").read_text())
no_sun_features = feature_groups["ablation"]["baseline_no_sun"]
solar_features = feature_groups["ablation"]["with_sun_trailing"]
train_mask = data["split"].eq("train")
valid_mask = data["split"].eq("valid")
test_mask = data["split"].eq("test")
for name, mask in [("Training", train_mask), ("Validation", valid_mask), ("Test", test_mask)]:
    group = data.loc[mask]
    print(f"{name}: {len(group):,} observations; {group.norad_cat_id.nunique():,} objects; positive rate {group[TARGET].mean():.4f}")
cohorts = [set(data.loc[mask, "norad_cat_id"]) for mask in [train_mask, valid_mask, test_mask]]
assert all(not cohorts[i] & cohorts[j] for i in range(3) for j in range(i+1, 3))
assert set(data.loc[train_mask, "obs_date"].dt.year).issubset(set(range(2015, 2021)))
assert data.loc[valid_mask, "obs_date"].dt.year.eq(2022).all()
assert data.loc[test_mask, "obs_date"].dt.year.ge(2024).all()
print("Verified plan periods, object-disjoint splits, and no use of the 2021/2023 gap years.")
y_train = data.loc[train_mask, TARGET].astype("int8")
groups_train = data.loc[train_mask, "norad_cat_id"]
cv_folds = list(GroupKFold(n_splits=5).split(data.loc[train_mask], y_train, groups_train))
for fit_rows, score_rows in cv_folds:
    assert not set(groups_train.iloc[fit_rows]) & set(groups_train.iloc[score_rows])
DECISION_THRESHOLD = 0.5
RUN_TEST_EVALUATION = True
training_results, evaluation_results = {}, {}


Training: 122,859 observations; 3,202 objects; positive rate 0.0519
Validation: 4,954 observations; 612 objects; positive rate 0.2990
Test: 6,921 observations; 508 objects; positive rate 0.2094
Verified plan periods, object-disjoint splits, and no use of the 2021/2023 gap years.


In [6]:
def make_model(columns, algorithm):
    numeric = data.loc[train_mask, columns].select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical = [column for column in columns if column not in numeric]
    numeric_steps = [("imputer", SimpleImputer(strategy="median", add_indicator=True))]
    if algorithm == "logistic":
        numeric_steps.append(("scale", StandardScaler()))
    preprocessor = ColumnTransformer([
        ("numeric", Pipeline(numeric_steps), numeric),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=algorithm == "logistic")),
        ]), categorical),
    ], sparse_threshold=0.3 if algorithm == "logistic" else 0)
    if algorithm == "logistic":
        estimator = LogisticRegression(class_weight="balanced", max_iter=1000, solver="lbfgs", random_state=SEED)
    else:
        estimator = HistGradientBoostingClassifier(
            learning_rate=0.1, max_iter=100, max_leaf_nodes=31, min_samples_leaf=20,
            l2_regularization=0.0, class_weight="balanced", early_stopping=False,
            random_state=SEED,
        )
    return Pipeline([("preprocess", preprocessor), ("classifier", estimator)])


def fit_and_report(columns, algorithm, name):
    model = make_model(columns, algorithm)
    X_train = data.loc[train_mask, columns]
    with threadpool_limits(limits=4):
        scores = cross_validate(model, X_train, y_train, cv=cv_folds,
            scoring={"accuracy": "accuracy", "average_precision": "average_precision", "roc_auc": "roc_auc", "brier": "neg_brier_score"},
            n_jobs=1, error_score="raise")
        model.fit(X_train, y_train)
    print(f"{name}: {len(columns)} inputs; {len(X_train):,} training observations; {groups_train.nunique():,} objects")
    training_results[name] = {}
    for metric in ["accuracy", "average_precision", "roc_auc", "brier"]:
        values = scores["test_" + metric] * (-1 if metric == "brier" else 1)
        training_results[name][metric] = {"mean": float(values.mean()), "std": float(values.std())}
        print(f"Grouped training CV {metric}: {values.mean():.4f} +/- {values.std():.4f}")
    return model


def evaluate(model, columns, mask, name, cohort):
    labels = data.loc[mask, TARGET]
    with threadpool_limits(limits=4):
        probability = model.predict_proba(data.loc[mask, columns])[:, 1]
    prediction = (probability >= DECISION_THRESHOLD).astype("int8")
    metrics = {
        "Accuracy": accuracy_score(labels, prediction),
        "Balanced accuracy": balanced_accuracy_score(labels, prediction),
        "Average precision": average_precision_score(labels, probability),
        "ROC AUC": roc_auc_score(labels, probability),
        "Precision": precision_score(labels, prediction, zero_division=0),
        "Recall": recall_score(labels, prediction, zero_division=0),
        "Brier score": brier_score_loss(labels, probability),
        "Training-majority baseline accuracy": accuracy_score(labels, np.full(len(labels), int(y_train.mean() >= 0.5))),
        "Training-rate baseline Brier score": brier_score_loss(labels, np.full(len(labels), y_train.mean())),
    }
    evaluation_results[(name, cohort)] = metrics
    print(f"{name} - {cohort}, threshold {DECISION_THRESHOLD:.2f}")
    print(f"Observations: {len(labels):,}; positive rate: {labels.mean():.4f}")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")
    print("Confusion matrix (rows actual 0/1; columns predicted 0/1):")
    print(confusion_matrix(labels, prediction, labels=[0, 1]))


## 1. Baseline models

These are the standard reference points for a binary classifier. The majority-class baseline always predicts the most common training class. The training-rate baseline always outputs the training positive rate as a probability and converts it to a class at threshold 0.5. Both are fitted from the training cohort only and use no features. Their purpose is to show the minimum performance a candidate model must beat.

### Training, validation, and test performance

The same metrics and cohorts are used for all splits. The test cohort is the reused-panel development holdout described above, not an untouched final test. Constant-probability baselines have no meaningful ROC ranking, so ROC AUC is reported as undefined; Brier score evaluates their probability quality.


In [7]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             average_precision_score, precision_score, recall_score,
                             brier_score_loss, confusion_matrix)

majority_class = int(y_train.mean() >= 0.5)
training_rate = float(y_train.mean())
baseline_results = {}

def evaluate_baselines(mask, cohort):
    labels = data.loc[mask, TARGET]
    majority_probability = np.full(len(labels), majority_class, dtype=float)
    rate_probability = np.full(len(labels), training_rate, dtype=float)
    rows = []
    for name, probability in [
        ("Majority-class baseline", majority_probability),
        ("Training-rate probability baseline", rate_probability),
    ]:
        prediction = (probability >= DECISION_THRESHOLD).astype("int8")
        metrics = {
            "Accuracy": accuracy_score(labels, prediction),
            "Balanced accuracy": balanced_accuracy_score(labels, prediction),
            "Average precision": average_precision_score(labels, probability),
            "ROC AUC": np.nan,
            "Precision": precision_score(labels, prediction, zero_division=0),
            "Recall": recall_score(labels, prediction, zero_division=0),
            "Brier score": brier_score_loss(labels, probability),
        }
        baseline_results[(name, cohort)] = metrics
        rows.append({"Model": name, "Cohort": cohort, **metrics})
        print(f"{name} - {cohort}, threshold {DECISION_THRESHOLD:.2f}")
        for metric, value in metrics.items():
            print(f"{metric}: {'undefined' if pd.isna(value) else f'{value:.4f}'}")
        print("Confusion matrix (rows actual 0/1; columns predicted 0/1):")
        print(confusion_matrix(labels, prediction, labels=[0, 1]))
    return rows

baseline_table = []
baseline_table.extend(evaluate_baselines(train_mask, "Training"))
baseline_table.extend(evaluate_baselines(valid_mask, "Validation"))
if RUN_TEST_EVALUATION:
    baseline_table.extend(evaluate_baselines(test_mask, "Test"))
else:
    print("Baseline test scoring disabled. Run the notebook runner with --include-test for explicit evaluation.")
baseline_table = pd.DataFrame(baseline_table)


Majority-class baseline - Training, threshold 0.50
Accuracy: 0.9481
Balanced accuracy: 0.5000
Average precision: 0.0519
ROC AUC: undefined
Precision: 0.0000
Recall: 0.0000
Brier score: 0.0519
Confusion matrix (rows actual 0/1; columns predicted 0/1):
[[116482      0]
 [  6377      0]]
Training-rate probability baseline - Training, threshold 0.50
Accuracy: 0.9481
Balanced accuracy: 0.5000
Average precision: 0.0519
ROC AUC: undefined
Precision: 0.0000
Recall: 0.0000
Brier score: 0.0492
Confusion matrix (rows actual 0/1; columns predicted 0/1):
[[116482      0]
 [  6377      0]]
Majority-class baseline - Validation, threshold 0.50
Accuracy: 0.7010
Balanced accuracy: 0.5000
Average precision: 0.2990
ROC AUC: undefined
Precision: 0.0000
Recall: 0.0000
Brier score: 0.2990
Confusion matrix (rows actual 0/1; columns predicted 0/1):
[[3473    0]
 [1481    0]]
Training-rate probability baseline - Validation, threshold 0.50
Accuracy: 0.7010
Balanced accuracy: 0.5000
Average precision: 0.2990
ROC 

## 1. Logistic regression without solar information


### Model description

Class-balanced logistic regression with L2 regularization, C=1, lbfgs solver, and max_iter=1000. Numeric inputs receive median imputation, missing indicators, and standardization. Categorical inputs receive missing-category imputation and one-hot encoding.

Uses 20 inputs: object, age, and orbit features; all solar/geomagnetic inputs excluded. Preprocessing is fitted separately inside each training fold. All four models use the same objects, CV folds, and threshold 0.5.


### Training performance

Five-fold cross-validation grouped by object on the 70% training cohort. Report fold mean and standard deviation for average precision, ROC AUC, and Brier score. These are held-out fold scores, not in-sample accuracy. Refit on the full training cohort afterward.


In [8]:
model_1 = fit_and_report(no_sun_features, "logistic", "Logistic regression without solar information")


Logistic regression without solar information: 20 inputs; 122,859 training observations; 3,202 objects
Grouped training CV accuracy: 0.9135 +/- 0.0190
Grouped training CV average_precision: 0.7488 +/- 0.0372
Grouped training CV roc_auc: 0.9656 +/- 0.0075
Grouped training CV brier: 0.0660 +/- 0.0115


### Validation performance

Evaluate on the 15% validation cohort without refitting preprocessing or adjusting the threshold.


In [9]:
evaluate(model_1, no_sun_features, valid_mask, "Logistic regression without solar information", "Validation")


Logistic regression without solar information - Validation, threshold 0.50
Observations: 4,954; positive rate: 0.2990
Accuracy: 0.8777
Balanced accuracy: 0.8645
Average precision: 0.8801
ROC AUC: 0.9444
Precision: 0.7753
Recall: 0.8319
Brier score: 0.0882
Training-majority baseline accuracy: 0.7010
Training-rate baseline Brier score: 0.2706
Confusion matrix (rows actual 0/1; columns predicted 0/1):
[[3116  357]
 [ 249 1232]]


### Test performance

Evaluate on the 15% development test cohort with the same fixed settings. This is a reused-panel comparison, not an untouched final evaluation. Routine Run All skips test scoring.


In [10]:
if RUN_TEST_EVALUATION:
    evaluate(model_1, no_sun_features, test_mask, "Logistic regression without solar information", "Test")
else:
    print("Test scoring disabled. Run the notebook runner with --include-test for explicit evaluation.")


Logistic regression without solar information - Test, threshold 0.50
Observations: 6,921; positive rate: 0.2094
Accuracy: 0.8853
Balanced accuracy: 0.8303
Average precision: 0.8292
ROC AUC: 0.9368
Precision: 0.7217
Recall: 0.7357
Brier score: 0.0830
Training-majority baseline accuracy: 0.7906
Training-rate baseline Brier score: 0.1903
Confusion matrix (rows actual 0/1; columns predicted 0/1):
[[5061  411]
 [ 383 1066]]


## 2. Logistic regression with solar information


### Model description

Class-balanced logistic regression with L2 regularization, C=1, lbfgs solver, and max_iter=1000. Numeric inputs receive median imputation, missing indicators, and standardization. Categorical inputs receive missing-category imputation and one-hot encoding.

Uses 52 inputs: object, age, and orbit features plus 32 solar/geomagnetic inputs. Preprocessing is fitted separately inside each training fold. All four models use the same objects, CV folds, and threshold 0.5.


### Training performance

Five-fold cross-validation grouped by object on the 70% training cohort. Report fold mean and standard deviation for average precision, ROC AUC, and Brier score. These are held-out fold scores, not in-sample accuracy. Refit on the full training cohort afterward.


In [11]:
model_2 = fit_and_report(solar_features, "logistic", "Logistic regression with solar information")


Logistic regression with solar information: 52 inputs; 122,859 training observations; 3,202 objects
Grouped training CV accuracy: 0.9147 +/- 0.0195
Grouped training CV average_precision: 0.7464 +/- 0.0413
Grouped training CV roc_auc: 0.9658 +/- 0.0073
Grouped training CV brier: 0.0656 +/- 0.0123


### Validation performance

Evaluate on the 15% validation cohort without refitting preprocessing or adjusting the threshold.


In [12]:
evaluate(model_2, solar_features, valid_mask, "Logistic regression with solar information", "Validation")


Logistic regression with solar information - Validation, threshold 0.50
Observations: 4,954; positive rate: 0.2990
Accuracy: 0.8644
Balanced accuracy: 0.8574
Average precision: 0.8794
ROC AUC: 0.9413
Precision: 0.7409
Recall: 0.8400
Brier score: 0.0934
Training-majority baseline accuracy: 0.7010
Training-rate baseline Brier score: 0.2706
Confusion matrix (rows actual 0/1; columns predicted 0/1):
[[3038  435]
 [ 237 1244]]


### Test performance

Evaluate on the 15% development test cohort with the same fixed settings. This is a reused-panel comparison, not an untouched final evaluation. Routine Run All skips test scoring.


In [13]:
if RUN_TEST_EVALUATION:
    evaluate(model_2, solar_features, test_mask, "Logistic regression with solar information", "Test")
else:
    print("Test scoring disabled. Run the notebook runner with --include-test for explicit evaluation.")


Logistic regression with solar information - Test, threshold 0.50
Observations: 6,921; positive rate: 0.2094
Accuracy: 0.8278
Balanced accuracy: 0.8337
Average precision: 0.7949
ROC AUC: 0.9154
Precision: 0.5587
Recall: 0.8440
Brier score: 0.1228
Training-majority baseline accuracy: 0.7906
Training-rate baseline Brier score: 0.1903
Confusion matrix (rows actual 0/1; columns predicted 0/1):
[[4506  966]
 [ 226 1223]]


## 3. Histogram gradient boosting without solar information


### Model description

Class-balanced histogram gradient boosting with 100 iterations, learning rate 0.1, 31 maximum leaves, minimum 20 samples per leaf, and L2 regularization 0. Numeric inputs receive median imputation and missing indicators; categorical inputs receive missing-category imputation and dense one-hot encoding. No scaling is needed. Automatic early stopping is disabled to avoid an internal row-level split that ignores object groups.

Uses 20 inputs: object, age, and orbit features; all solar/geomagnetic inputs excluded. Preprocessing is fitted separately inside each training fold. All four models use the same objects, CV folds, and threshold 0.5.


### Training performance

Five-fold cross-validation grouped by object on the 70% training cohort. Report fold mean and standard deviation for average precision, ROC AUC, and Brier score. These are held-out fold scores, not in-sample accuracy. Refit on the full training cohort afterward.


In [14]:
model_3 = fit_and_report(no_sun_features, "boost", "Histogram gradient boosting without solar information")


Histogram gradient boosting without solar information: 20 inputs; 122,859 training observations; 3,202 objects
Grouped training CV accuracy: 0.9746 +/- 0.0042
Grouped training CV average_precision: 0.9428 +/- 0.0073
Grouped training CV roc_auc: 0.9946 +/- 0.0010
Grouped training CV brier: 0.0186 +/- 0.0027


### Validation performance

Evaluate on the 15% validation cohort without refitting preprocessing or adjusting the threshold.


In [15]:
evaluate(model_3, no_sun_features, valid_mask, "Histogram gradient boosting without solar information", "Validation")


Histogram gradient boosting without solar information - Validation, threshold 0.50
Observations: 4,954; positive rate: 0.2990
Accuracy: 0.9287
Balanced accuracy: 0.9037
Average precision: 0.9576
ROC AUC: 0.9785
Precision: 0.9135
Recall: 0.8413
Brier score: 0.0568
Training-majority baseline accuracy: 0.7010
Training-rate baseline Brier score: 0.2706
Confusion matrix (rows actual 0/1; columns predicted 0/1):
[[3355  118]
 [ 235 1246]]


### Test performance

Evaluate on the 15% development test cohort with the same fixed settings. This is a reused-panel comparison, not an untouched final evaluation. Routine Run All skips test scoring.


In [16]:
if RUN_TEST_EVALUATION:
    evaluate(model_3, no_sun_features, test_mask, "Histogram gradient boosting without solar information", "Test")
else:
    print("Test scoring disabled. Run the notebook runner with --include-test for explicit evaluation.")


Histogram gradient boosting without solar information - Test, threshold 0.50
Observations: 6,921; positive rate: 0.2094
Accuracy: 0.9198
Balanced accuracy: 0.9128
Average precision: 0.9350
ROC AUC: 0.9757
Precision: 0.7605
Recall: 0.9006
Brier score: 0.0602
Training-majority baseline accuracy: 0.7906
Training-rate baseline Brier score: 0.1903
Confusion matrix (rows actual 0/1; columns predicted 0/1):
[[5061  411]
 [ 144 1305]]


## 4. Histogram gradient boosting with solar information


### Model description

Class-balanced histogram gradient boosting with 100 iterations, learning rate 0.1, 31 maximum leaves, minimum 20 samples per leaf, and L2 regularization 0. Numeric inputs receive median imputation and missing indicators; categorical inputs receive missing-category imputation and dense one-hot encoding. No scaling is needed. Automatic early stopping is disabled to avoid an internal row-level split that ignores object groups.

Uses 52 inputs: object, age, and orbit features plus 32 solar/geomagnetic inputs. Preprocessing is fitted separately inside each training fold. All four models use the same objects, CV folds, and threshold 0.5.


### Training performance

Five-fold cross-validation grouped by object on the 70% training cohort. Report fold mean and standard deviation for average precision, ROC AUC, and Brier score. These are held-out fold scores, not in-sample accuracy. Refit on the full training cohort afterward.


In [17]:
model_4 = fit_and_report(solar_features, "boost", "Histogram gradient boosting with solar information")


Histogram gradient boosting with solar information: 52 inputs; 122,859 training observations; 3,202 objects
Grouped training CV accuracy: 0.9799 +/- 0.0031
Grouped training CV average_precision: 0.9550 +/- 0.0075
Grouped training CV roc_auc: 0.9957 +/- 0.0009
Grouped training CV brier: 0.0150 +/- 0.0020


### Validation performance

Evaluate on the 15% validation cohort without refitting preprocessing or adjusting the threshold.


In [18]:
evaluate(model_4, solar_features, valid_mask, "Histogram gradient boosting with solar information", "Validation")


Histogram gradient boosting with solar information - Validation, threshold 0.50
Observations: 4,954; positive rate: 0.2990
Accuracy: 0.9174
Balanced accuracy: 0.8716
Average precision: 0.9588
ROC AUC: 0.9793
Precision: 0.9573
Recall: 0.7576
Brier score: 0.0681
Training-majority baseline accuracy: 0.7010
Training-rate baseline Brier score: 0.2706
Confusion matrix (rows actual 0/1; columns predicted 0/1):
[[3423   50]
 [ 359 1122]]


### Test performance

Evaluate on the 15% development test cohort with the same fixed settings. This is a reused-panel comparison, not an untouched final evaluation. Routine Run All skips test scoring.


In [19]:
if RUN_TEST_EVALUATION:
    evaluate(model_4, solar_features, test_mask, "Histogram gradient boosting with solar information", "Test")
else:
    print("Test scoring disabled. Run the notebook runner with --include-test for explicit evaluation.")


Histogram gradient boosting with solar information - Test, threshold 0.50
Observations: 6,921; positive rate: 0.2094
Accuracy: 0.9416
Balanced accuracy: 0.9035
Average precision: 0.9391
ROC AUC: 0.9770
Precision: 0.8778
Recall: 0.8378
Brier score: 0.0458
Training-majority baseline accuracy: 0.7906
Training-rate baseline Brier score: 0.1903
Confusion matrix (rows actual 0/1; columns predicted 0/1):
[[5303  169]
 [ 235 1214]]


## Model comparison

All scores below come from the current 70/15/15 split. Test results are descriptive follow-up results; use training CV and validation for any later selection.


In [20]:
comparison = pd.DataFrame([
    {"Model": name, "Cohort": cohort, **metrics}
    for (name, cohort), metrics in evaluation_results.items()
])
print(comparison[["Model", "Cohort", "Accuracy", "Average precision", "ROC AUC", "Brier score"]].to_string(index=False, float_format=lambda x: f"{x:.4f}"))


                                                Model     Cohort  Accuracy  Average precision  ROC AUC  Brier score
        Logistic regression without solar information Validation    0.8777             0.8801   0.9444       0.0882
        Logistic regression without solar information       Test    0.8853             0.8292   0.9368       0.0830
           Logistic regression with solar information Validation    0.8644             0.8794   0.9413       0.0934
           Logistic regression with solar information       Test    0.8278             0.7949   0.9154       0.1228
Histogram gradient boosting without solar information Validation    0.9287             0.9576   0.9785       0.0568
Histogram gradient boosting without solar information       Test    0.9198             0.9350   0.9757       0.0602
   Histogram gradient boosting with solar information Validation    0.9174             0.9588   0.9793       0.0681
   Histogram gradient boosting with solar information       Test    0.94

## Selected model

Select using validation average precision as the primary metric, with Brier score as the probability-quality check. Training cross-validation is used as a stability check. The test metrics are excluded from selection because they have already been inspected. This rule favors ranking performance while checking that predicted probabilities are useful.


In [21]:
# Compare only validation results and grouped training CV; test scores do not enter selection.
candidate_names = [
    "Logistic regression without solar information",
    "Logistic regression with solar information",
    "Histogram gradient boosting without solar information",
    "Histogram gradient boosting with solar information",
]
selection_rows = []
for candidate in candidate_names:
    validation_metrics = evaluation_results[(candidate, "Validation")]
    cv = training_results[candidate]
    selection_rows.append({
        "Model": candidate,
        "CV average precision": cv["average_precision"]["mean"],
        "CV AP std": cv["average_precision"]["std"],
        "Validation average precision": validation_metrics["Average precision"],
        "Validation Brier score": validation_metrics["Brier score"],
    })
selection_table = pd.DataFrame(selection_rows).sort_values(
    ["Validation average precision", "Validation Brier score"],
    ascending=[False, True],
).reset_index(drop=True)
selected_model_name = selection_table.loc[0, "Model"]
selected_model, selected_features = {
    candidate_names[0]: (model_1, no_sun_features),
    candidate_names[1]: (model_2, solar_features),
    candidate_names[2]: (model_3, no_sun_features),
    candidate_names[3]: (model_4, solar_features),
}[selected_model_name]
print(selection_table.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print(f"Selected by validation rule: {selected_model_name}")


                                                Model  CV average precision  CV AP std  Validation average precision  Validation Brier score
   Histogram gradient boosting with solar information                0.9550     0.0075                        0.9588                  0.0681
Histogram gradient boosting without solar information                0.9428     0.0073                        0.9576                  0.0568
        Logistic regression without solar information                0.7488     0.0372                        0.8801                  0.0882
           Logistic regression with solar information                0.7464     0.0413                        0.8794                  0.0934
Selected by validation rule: Histogram gradient boosting with solar information


## Error analysis

Inspect false positives and false negatives for the selected model by orbit altitude, object type, radar size class, and solar activity. Tables include row and object counts so sparse groups are visible. Use validation for diagnosis; test tables are descriptive because this test cohort has already been inspected. The solar-activity analysis is omitted for a model without solar inputs, but labels and post-observation group variables remain diagnostic only.

In [22]:
from sklearn.metrics import confusion_matrix

def error_analysis(mask, cohort_name):
    subset = data.loc[mask].copy()
    probability = selected_model.predict_proba(subset[selected_features])[:, 1]
    subset["probability"] = probability
    subset["prediction"] = (probability >= DECISION_THRESHOLD).astype("int8")
    subset["error"] = subset["prediction"] != subset[TARGET]
    subset["error_type"] = np.select(
        [(subset.prediction == 1) & (subset[TARGET] == 0),
         (subset.prediction == 0) & (subset[TARGET] == 1)],
        ["false_positive", "false_negative"], default="correct",
    )
    print(f"{cohort_name}: {selected_model_name}; threshold={DECISION_THRESHOLD:.2f}")
    print("Confusion matrix (rows actual 0/1; columns predicted 0/1):")
    print(confusion_matrix(subset[TARGET], subset.prediction, labels=[0, 1]))
    if "perigee_km_asof" in subset:
        subset["perigee_band"] = pd.cut(
            subset["perigee_km_asof"],
            bins=[-np.inf, 200, 400, 600, 1000, np.inf],
            labels=["<=200 km", "200-400 km", "400-600 km", "600-1000 km", ">1000 km"],
        )
    if "f107_adj_mean_81d" in subset:
        subset["solar_activity_tercile"] = pd.qcut(
            subset["f107_adj_mean_81d"], q=3,
            labels=["low", "medium", "high"], duplicates="drop",
        )
    groups = [column for column in ["perigee_band", "object_type", "rcs_size", "solar_activity_tercile"] if column in subset]
    tables = {}
    for column in groups:
        table = subset.groupby(column, observed=True, dropna=False).agg(
            rows=(TARGET, "size"),
            objects=("norad_cat_id", "nunique"),
            positive_rate=(TARGET, "mean"),
            error_rate=("error", "mean"),
            false_positives=("error_type", lambda values: int((values == "false_positive").sum())),
            false_negatives=("error_type", lambda values: int((values == "false_negative").sum())),
        ).reset_index()
        table.insert(0, "cohort", cohort_name)
        tables[column] = table
        print(f"\nError analysis by {column}:")
        print(table.to_string(index=False, float_format=lambda value: f"{value:.3f}"))
    return tables

validation_error_tables = error_analysis(valid_mask, "Validation")
if RUN_TEST_EVALUATION:
    test_error_tables = error_analysis(test_mask, "Development test")


Validation: Histogram gradient boosting with solar information; threshold=0.50
Confusion matrix (rows actual 0/1; columns predicted 0/1):
[[3423   50]
 [ 359 1122]]

Error analysis by perigee_band:
    cohort perigee_band  rows  objects  positive_rate  error_rate  false_positives  false_negatives
Validation     <=200 km    73       14          0.452       0.082                6                0
Validation   200-400 km  1293      274          0.326       0.022                9               19
Validation   400-600 km  3588      460          0.286       0.105               35              340

Error analysis by object_type:
    cohort object_type  rows  objects  positive_rate  error_rate  false_positives  false_negatives
Validation      DEBRIS  3691      491          0.385       0.104               43              341
Validation ROCKET BODY   937       83          0.057       0.021                7               13
Validation     UNKNOWN   326       38          0.018       0.015         

## Limitations

- The target is a binary, 365-day outcome among objects with recorded re-entry. Objects still in orbit are excluded, so this does not directly establish performance for the full active debris catalogue.
- Observation rows were sampled over object lifetimes. This is not a prospective, calendar-based forecast evaluation; age and sampling may make the task easier than live use.
- The 70/15/15 split keeps objects separate, but prior results on the same panel were already viewed. Treat the current test as a development holdout, not an untouched final test.
- Space-weather features are predictive associations. This comparison does not show that solar activity causes re-entry or that the small solar-feature gain will persist in another period.
- Accuracy depends on the 0.5 decision threshold and roughly balanced class weights. A deployed alert threshold should reflect the relative costs of missed re-entries and false alarms, selected using validation.
- Grouped CV uncertainty is summarized across five folds; error-analysis subgroups can still be small and observations from one object are not independent. Read subgroup counts and avoid strong claims from sparse categories.
- The selected model and its scores need external or future-time validation before operational use.

## Model summary

| Model | Algorithm | Features used |
|---|---|---|
| Logistic regression without solar information | Class-balanced logistic regression (L2, C=1) | 20: object characteristics (6), age (1), as-of orbit (13) |
| Logistic regression with solar information | Class-balanced logistic regression (L2, C=1) | 52: object characteristics (6), age (1), as-of orbit (13), trailing solar/geomagnetic weather (32) |
| Histogram gradient boosting without solar information | Class-balanced histogram gradient boosting | 20: object characteristics (6), age (1), as-of orbit (13) |
| Histogram gradient boosting with solar information | Class-balanced histogram gradient boosting | 52: object characteristics (6), age (1), as-of orbit (13), trailing solar/geomagnetic weather (32) |

Object characteristics include type, debris indicator, radar size class/value, country, and catalogue inclination. As-of orbit features include altitude, perigee, apogee, orbital shape and motion, BSTAR, perigee decay rate, and element-set age. Weather features summarize F10.7 solar flux, sunspot number, and geomagnetic indices over trailing windows.


## Logistic regression feature analysis

Permutation importance on the validation cohort measures how much each model's average precision and Brier score change when one raw input is shuffled. Group ablations shuffle related inputs together, preserving relationships within each group. These are diagnostics, not causal effects; correlated inputs can share importance. Test data is not used.


In [1]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, brier_score_loss

FEATURE_GROUPS = {
    "Object characteristics": ["inclination_deg", "rcs_size", "rcs_value", "object_type", "is_debris", "country"],
    "Object age": ["age_days"],
    "As-of orbit": [c for c in no_sun_features if c not in {"inclination_deg", "rcs_size", "rcs_value", "object_type", "is_debris", "country", "age_days"}],
    "Solar and geomagnetic weather": feature_groups["features"]["space_weather_trailing"],
}

def logistic_feature_analysis(model, feature_columns, model_name, repeats=5, seed=20261116):
    X_valid = data.loc[valid_mask, feature_columns]
    y_valid = data.loc[valid_mask, TARGET]
    base_probability = model.predict_proba(X_valid)[:, 1]
    base_ap = average_precision_score(y_valid, base_probability)
    base_brier = brier_score_loss(y_valid, base_probability)
    rng = np.random.default_rng(seed)
    rows = []
    available_groups = {name: [c for c in cols if c in feature_columns] for name, cols in FEATURE_GROUPS.items()}
    for group_name, columns in available_groups.items():
        if not columns:
            continue
        ap_drops, brier_rises = [], []
        for _ in range(repeats):
            order = rng.permutation(len(X_valid))
            permuted = X_valid.copy()
            permuted.loc[:, columns] = X_valid.iloc[order][columns].to_numpy()
            probability = model.predict_proba(permuted)[:, 1]
            ap_drops.append(base_ap - average_precision_score(y_valid, probability))
            brier_rises.append(brier_score_loss(y_valid, probability) - base_brier)
        rows.append({"Model": model_name, "Feature group": group_name, "Features shuffled": len(columns),
                     "AP drop (mean)": np.mean(ap_drops), "AP drop (SD)": np.std(ap_drops),
                     "Brier increase (mean)": np.mean(brier_rises)})
    group_table = pd.DataFrame(rows).sort_values("AP drop (mean)", ascending=False)
    print(f"{model_name}: validation AP={base_ap:.4f}; Brier={base_brier:.4f}")
    print(group_table.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

    individual = permutation_importance(
        model, X_valid, y_valid, scoring={"average_precision": "average_precision", "neg_brier": "neg_brier_score"},
        n_repeats=repeats, random_state=seed, n_jobs=1,
    )
    individual_table = pd.DataFrame({
        "Feature": feature_columns,
        "AP drop (mean)": individual["average_precision"].importances_mean,
        "AP drop (SD)": individual["average_precision"].importances_std,
        "Brier increase (mean)": individual["neg_brier"].importances_mean,
    }).sort_values("AP drop (mean)", ascending=False)
    print("Top individual inputs on validation:")
    print(individual_table.head(15).to_string(index=False, float_format=lambda value: f"{value:.4f}"))
    return group_table, individual_table

logistic_no_sun_group_importance, logistic_no_sun_feature_importance = logistic_feature_analysis(
    model_1, no_sun_features, "Logistic regression without solar information", seed=20261116)
logistic_sun_group_importance, logistic_sun_feature_importance = logistic_feature_analysis(
    model_2, solar_features, "Logistic regression with solar information", seed=20261117)


Logistic regression without solar information: validation AP=0.8801; Brier=0.0882
                                        Model          Feature group  Features shuffled  AP drop (mean)  AP drop (SD)  Brier increase (mean)
Logistic regression without solar information            As-of orbit                 13          0.5189        0.0087                 0.2059
Logistic regression without solar information Object characteristics                  6          0.1132        0.0025                 0.0580
Logistic regression without solar information             Object age                  1         -0.0011        0.0004                -0.0008
Top individual inputs on validation:
             Feature  AP drop (mean)  AP drop (SD)  Brier increase (mean)
     period_min_asof          0.5751        0.0032                0.2752
         mean_motion          0.5745        0.0043                0.3014
      apogee_km_asof          0.2444        0.0102                0.1096
         sma_km_asof    